In [34]:
import numpy as np
import re 
import tensorflow as tf
from sklearn.model_selection import train_test_split

In [35]:
# DATASET_PATH = 'ind/ind.txt'

def preprocess_sentence(w):
    w = w.lower()
    w = re.sub(r"([?!.,])", r"\1 ", w) #Halo!semua => Halo! semua
    w = re.sub(r'[" "]+', r" ", w) # Halo  semua => Halo semua
    w = re.sub(r'[^a-zA-Z?!.,]', r" ", w) #menghilangkan char spesial lain
    w = w.strip() #menghilangkan spasi di awal + akhir kalimat
    w = '<start> ' + w + ' <end>'
    return w
    
print(preprocess_sentence(" HALO!Semua  nama saya wilson"))

<start> halo! semua nama saya wilson <end>


In [36]:
def load_dataset(path, num_examples=None):
    with open(path, 'r', encoding='utf-8') as file:
        lines = file.read().strip().split('\n')
    parts = [line.split('\t') for line in lines[:num_examples]]
    
    input_data = [preprocess_sentence(part[0]) for part in parts]
    target_data = [preprocess_sentence(part[1]) for part in parts]
    
    return input_data, target_data

path_to_file = 'ind/ind.txt'
input_data, target_data = load_dataset(path_to_file, 1000)

print(f"input data: {input_data}")
print(f"target data: {target_data}")

input data: ['<start> hi. <end>', '<start> run! <end>', '<start> run. <end>', '<start> who? <end>', '<start> wow! <end>', '<start> wow! <end>', '<start> fire! <end>', '<start> fire! <end>', '<start> fire! <end>', '<start> help! <end>', '<start> jump! <end>', '<start> jump. <end>', '<start> stop! <end>', '<start> stop! <end>', '<start> wait! <end>', '<start> wait. <end>', '<start> hello. <end>', '<start> hurry! <end>', '<start> hurry! <end>', '<start> hurry! <end>', '<start> i see. <end>', '<start> i see. <end>', '<start> i see. <end>', '<start> i see. <end>', '<start> i see. <end>', '<start> i see. <end>', '<start> i won! <end>', '<start> i won. <end>', '<start> oh no! <end>', '<start> relax. <end>', '<start> relax. <end>', '<start> shoot! <end>', '<start> smile. <end>', '<start> attack! <end>', '<start> cheers! <end>', '<start> freeze! <end>', '<start> get up. <end>', '<start> got it! <end>', '<start> got it? <end>', '<start> he ran. <end>', '<start> i fell. <end>', '<start> i know. <

In [37]:
tokenizer = tf.keras.preprocessing.text.Tokenizer(filters='')
tokenizer.fit_on_texts(input_data + target_data)

input_tensor = tokenizer.texts_to_sequences(input_data)
target_tensor = tokenizer.texts_to_sequences(target_data)

max_length_targ = max(len(seq) for seq in target_tensor)
max_length_inp = max(len(seq) for seq in input_tensor)

input_tensor = tf.keras.preprocessing.sequence.pad_sequences(input_tensor, maxlen=max_length_inp, padding='post')

target_tensor = tf.keras.preprocessing.sequence.pad_sequences(target_tensor, maxlen=max_length_targ, padding='post')

vocab_size = len(tokenizer.word_index) + 1
print("Input tensor: ", input_tensor)
print("Target tensor: ", target_tensor)
print(tokenizer.word_index)

Input tensor:  [[  1 634   2 ...   0   0   0]
 [  1 635   2 ...   0   0   0]
 [  1 636   2 ...   0   0   0]
 ...
 [  1   9   5 ...   2   0   0]
 [  1   9  10 ...   2   0   0]
 [  1   9  10 ...   2   0   0]]
Target tensor:  [[   1 1032    2 ...    0    0    0]
 [   1  505    2 ...    0    0    0]
 [   1  505    2 ...    0    0    0]
 ...
 [   1   18    5 ... 1504    2    0]
 [   1 1505    2 ...    0    0    0]
 [   1   18   37 ...    2    0    0]]
{'<start>': 1, '<end>': 2, 'i': 3, 'aku': 4, 'tom': 5, 's': 6, 'm': 7, 'saya': 8, 'is': 9, 'it': 10, 'dia': 11, 't': 12, 'tidak': 13, 'a': 14, 'itu': 15, 'he': 16, 'you': 17, 'apa': 18, 'don': 19, 'can': 20, 'kamu': 21, 'ini': 22, 'me.': 23, 'jangan': 24, 'tom.': 25, 'bisa': 26, 'are': 27, 'we': 28, 'it.': 29, 'who': 30, 'di': 31, 'yang': 32, 'let': 33, 'like': 34, 'suka': 35, 'kita': 36, 'sedang': 37, 'that': 38, 'do': 39, 'kami': 40, 'sudah': 41, 'was': 42, 'have': 43, 'she': 44, 'apakah': 45, 'am': 46, 'you.': 47, 're': 48, 'siapa': 49, 'ad

In [38]:
input_tensor_train, input_tensor_val, target_tensor_train, target_tensor_val = train_test_split(input_tensor, target_tensor,test_size=0.2, random_state=42)

BUFFER_SIZE = len(input_tensor_train)
BATCH_SIZE = 64

dataset = tf.data.Dataset.from_tensor_slices((input_tensor_train, target_tensor_train)).shuffle(BUFFER_SIZE)
dataset = dataset.batch(BATCH_SIZE, drop_remainder=True)

print("Dataset: ", dataset)

for sample in dataset.take(1):
    input_sample, target_sample = sample
    print("Input Example: ", input_sample)
    print("Target Example: ", target_sample)

Dataset:  <BatchDataset element_spec=(TensorSpec(shape=(64, 7), dtype=tf.int32, name=None), TensorSpec(shape=(64, 7), dtype=tf.int32, name=None))>
Input Example:  tf.Tensor(
[[   1    3    7   95    2    0    0]
 [   1    3  207  883    2    0    0]
 [   1    5    6  289    2    0    0]
 [   1    3    7   14  806    2    0]
 [   1    3   20   12  464    2    0]
 [   1    3   34   58    2    0    0]
 [   1    3   34 1011    2    0    0]
 [   1   33    6  822    2    0    0]
 [   1   30   46  188    2    0    0]
 [   1    3    7  900    2    0    0]
 [   1  671   23    2    0    0    0]
 [   1   10   42  909    2    0    0]
 [   1   16  149  784    2    0    0]
 [   1    5    9  455    2    0    0]
 [   1    9    5  905    2    0    0]
 [   1   52    9  933    2    0    0]
 [   1  172    2    0    0    0    0]
 [   1   94  392    2    0    0    0]
 [   1   42   16  113    2    0    0]
 [   1   16  258    2    0    0    0]
 [   1    3  196   25    2    0    0]
 [   1  477   57   23    2  

In [40]:
class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_size, enc_units, batch_size):
        super(Encoder, self).__init__()
        self.batch_size = batch_size
        self.enc_units = enc_units
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_size)
        self.gru = tf.keras.layers.GRU(
            self.enc_units, 
            return_sequences=True, 
            return_state=True, 
            recurrent_initializer='glorot_uniform')

    def call(self, x, hidden):
        x = self.embedding(x)
        output, state = self.gru(x, initial_state=hidden)        
        return output, state

    def initialize_hidden_state(self):
        return tf.zeros((self.batch_size, self.enc_units))

embedding_dim = 256
units= 1024
BATCH_SIZE = 64
example_input_batch = input_tensor_train[:BATCH_SIZE]

encoder = Encoder(vocab_size, embedding_dim, units, BATCH_SIZE)

sample_hidden = encoder.initialize_hidden_state()
sample_output, sample_hidden = encoder(example_input_batch, sample_hidden)

print("Encoder output size: {}".format(sample_output.shape))
print("Encoder hidden state shape: {}". format(sample_hidden.shape))

Encoder output size: (64, 7, 1024)
Encoder hidden state shape: (64, 1024)
